##### How to handle scientific notation values?

|   Scientific Value   |      Value         |
|----------------------|--------------------|
| 1.202040336829001E17 | 120204033682900096 |

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType
from pyspark.sql.functions import col, format_number

##### 1) Keep precise values: cast to DecimalType

In [0]:
from pyspark.sql.functions import expr

df1 = spark.range(1).select(
    # DECIMAL(38,20): 38 total digits, 20 digits after decimal point
    expr("CAST(1.0125000010125E-8 AS DECIMAL(38,20))").alias("x_dec")
)

display(df1)

df1.printSchema()

x_dec
1.012500001013E-8


root
 |-- x_dec: decimal(38,20) (nullable = true)



In [0]:
from pyspark.sql.functions import expr

df2 = spark.range(1).select(
    expr("CAST(1.0125000010125E-8 AS DECIMAL(38,20))").alias("x_dec")
)

df2.show()
df2.show(truncate=False)

df2.printSchema()

+--------------------+
|               x_dec|
+--------------------+
|0.000000010125000...|
+--------------------+

+----------------------+
|x_dec                 |
+----------------------+
|0.00000001012500001013|
+----------------------+

root
 |-- x_dec: decimal(38,20) (nullable = true)



     Even though you wrote:
     CAST(1.0125000010125E-8 AS DECIMAL(38,20))

     👉 Spark stores it correctly as DECIMAL, but
     👉 .display() (Databricks) formats small numbers in scientific notation

In [0]:
%sql
CREATE OR REPLACE TABLE decimal_data (
    id INT,
    x_dec DOUBLE
);

INSERT INTO decimal_data VALUES
(1, 1.0125000010125E-8),
(2, 2.5000000000000E-5),
(3, 0.123456789012345),
(4, 100.25);

SELECT * FROM decimal_data;

id,x_dec
1,1.0125000010125E-8
2,2.5E-5
3,0.123456789012345
4,100.25


In [0]:
from pyspark.sql.types import DecimalType
from pyspark.sql.functions import col

df3 = spark.table("decimal_data")

df4 = df3.withColumn(
    "x_dec_casted",
    col("x_dec").cast(DecimalType(38, 20))
)

df4.display()

id,x_dec,x_dec_casted
1,1.0125000010125E-8,1.012500001013E-8
2,2.5E-5,0.00002500000000000000
3,0.123456789012345,0.12345678901234500000
4,100.25,100.25000000000000000000


##### 2) Show or export without scientific notation (format as string)
- Increase precision (e.g., "%.20f") as needed.
- This is especially useful before writing CSV/TSV so values won’t appear in scientific notation.

In [0]:
%sql
CREATE OR REPLACE TABLE decimal_format_data (
    id INT,
    x DOUBLE
);

INSERT INTO decimal_format_data VALUES
(1, 1.0125000010125E-8),
(2, 2.3456789),
(3, 100.5),
(4, 0.000123456789);

SELECT * FROM decimal_format_data;

id,x
1,1.0125000010125E-8
2,2.3456789
3,100.5
4,1.23456789E-4


In [0]:
from pyspark.sql.functions import format_string, col

df5 = spark.table("decimal_format_data")

df_out = df5.withColumn(
    "x_str",
    format_string("%.15f", col("x"))
)

df_out.display()

id,x,x_str
1,1.0125000010125E-8,0.000000010125000
2,2.3456789,2.345678900000000
3,100.5,100.500000000000000
4,1.23456789E-4,0.000123456789000


**format_string("%.15f", col("x"))**
- `converts number → fixed decimal string`.
- `avoids scientific notation`.

**"%.15f"**
- `15 digits after decimal`.
- `you can change (e.g., %.10f, %.20f)`.

##### 3) scale to integers or keep decimals

**Option A: stay in Decimal**

In [0]:
%sql
CREATE TABLE decimal_scale_data (
    id INT,
    x DOUBLE
);

INSERT INTO decimal_scale_data VALUES
(1, 1.0125000010125E-8),
(2, 0.00000005),
(3, 0.123456789),
(4, 10.5);

SELECT * FROM decimal_scale_data;

id,x
1,1.0125000010125E-8
2,5.0E-8
3,0.123456789
4,10.5


In [0]:
from pyspark.sql.types import DecimalType
from pyspark.sql.functions import col

df6 = spark.table("decimal_scale_data")

dfD = df6.withColumn("x_dec", col("x").cast(DecimalType(38, 20))) \
         .withColumn("y_dec", (col("x_dec") * 100000000).cast(DecimalType(38, 0)))

dfD.display()

id,x,x_dec,y_dec
1,1.0125000010125E-8,1.012500001013E-8,1
2,5.0E-8,5.000000000000E-8,5
3,0.123456789,0.12345678900000000000,12345679
4,10.5,10.50000000000000000000,1050000000


**Option B: scale to integers (avoid floating ops)**

In [0]:
scale = 10**8

dfI = df6.withColumn(
    "x_scaled_int",
    (col("x").cast(DecimalType(38, 20)) * scale).cast("long")
)

dfI.display()

id,x,x_scaled_int
1,1.0125000010125E-8,1
2,5.0E-8,5
3,0.123456789,12345678
4,10.5,1050000000


##### 4) USE CASE 01

In [0]:
data = [(1, 1.20204033682900096E17),
        (2, 1.20204033682900096E17),
        (3, 1.20204033682900096E17),
        (4, 1.20204033682900096E17),
        (5, 1.20204033682900096E17),
        (6, 1.20204033682900096E17),
        (7, 1.20204033682900096E17),
        (8, 1.20204033682900096E17),
        (9, 1.20204033682900096E17)
        ]

schema = ["id", "productId"]

df_sct = spark.createDataFrame(data, schema)
display(df_sct)

id,productId
1,1.202040336829001E17
2,1.202040336829001E17
3,1.202040336829001E17
4,1.202040336829001E17
5,1.202040336829001E17
6,1.202040336829001E17
7,1.202040336829001E17
8,1.202040336829001E17
9,1.202040336829001E17


In [0]:
df_final = (df_sct \
    .withColumn("product_Id_long", col("productId").cast("long"))   
    .withColumn("product_Id", col("productId").cast("long").cast("string")))
display(df_final)

id,productId,product_Id_long,product_Id
1,1.202040336829001E17,120204033682900096,120204033682900096
2,1.202040336829001E17,120204033682900096,120204033682900096
3,1.202040336829001E17,120204033682900096,120204033682900096
4,1.202040336829001E17,120204033682900096,120204033682900096
5,1.202040336829001E17,120204033682900096,120204033682900096
6,1.202040336829001E17,120204033682900096,120204033682900096
7,1.202040336829001E17,120204033682900096,120204033682900096
8,1.202040336829001E17,120204033682900096,120204033682900096
9,1.202040336829001E17,120204033682900096,120204033682900096


In [0]:
from pyspark.sql.types import DecimalType

df_fixed = df_sct.withColumn(
    "productId_dec",
    col("productId").cast(DecimalType(38, 0))
)

df_fixed.display()

id,productId,productId_dec
1,1.202040336829001E17,120204033682900096
2,1.202040336829001E17,120204033682900096
3,1.202040336829001E17,120204033682900096
4,1.202040336829001E17,120204033682900096
5,1.202040336829001E17,120204033682900096
6,1.202040336829001E17,120204033682900096
7,1.202040336829001E17,120204033682900096
8,1.202040336829001E17,120204033682900096
9,1.202040336829001E17,120204033682900096


- `Spark treats this as DOUBLE`.
- `DOUBLE loses precision for large numbers`.
- `So even if you convert, the value may already be slightly wrong`.

##### BEST PRACTICE (No precision loss 🚀)
- `Define schema as STRING initially`.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

data = [
    (1, "120204033682900096"),
    (2, "120204033682900096")
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("productId", StringType(), True)
])

df_sct_01 = spark.createDataFrame(data, schema)
display(df_sct_01)

id,productId
1,120204033682900096
2,120204033682900096


In [0]:
from pyspark.sql.types import DecimalType

df_sct_final = df_sct_01.withColumn(
    "productId_dec",
    col("productId").cast(DecimalType(38, 0))
)

display(df_sct_final)

id,productId,productId_dec
1,120204033682900096,120204033682900096
2,120204033682900096,120204033682900096


##### 4) USE CASE 02

In [0]:
data = [(1,1),
        (2,12),
        (3,123),
        (4,1234),
        (5,12345),
        (6,123456),
        (7,1234567),
        (8,12345678),
        (9,123456789)
        ]

schema = StructType([StructField("id", IntegerType(), True), StructField("val", IntegerType(), True)])
 
df_frmt = spark.createDataFrame(data=data, schema=schema) \
    .withColumn("new_val", col("val")/98765432) \
    .select("id", "val", "new_val")
    
display(df_frmt)
df_frmt.printSchema()

id,val,new_val
1,1,1.0125000010125E-8
2,12,1.215000001215E-7
3,123,1.245375001245375E-6
4,1234,1.249425001249425E-5
5,12345,1.2499312512499313E-4
6,123456,0.001249992001249992
7,1234567,0.012499990887499991
8,12345678,0.124999989875
9,123456789,1.249999989875


root
 |-- id: integer (nullable = true)
 |-- val: integer (nullable = true)
 |-- new_val: double (nullable = true)



- Represented as **1.0125000010125E-8** and we call it **E to the power of** numbers.
- The `values` are presented in `exponential format` i.e. numbers with **E** `near the end`.
- if you are reading data from any Database via **JDBC** connection and the **datatype is DECIMAL** with **scale more than 6** then the value is **converted to exponential format** in Spark.

In [0]:
df_frmt_final = df_frmt.select("id", "val", format_number(col("new_val"),10).alias("new_val"))
display(df_frmt_final)
df_frmt_final.printSchema()

id,val,new_val
1,1,0.0000000101
2,12,0.0000001215
3,123,0.0000012454
4,1234,0.0000124943
5,12345,0.0001249931
6,123456,0.0012499920
7,1234567,0.0124999909
8,12345678,0.1249999899
9,123456789,1.2499999899


root
 |-- id: integer (nullable = true)
 |-- val: integer (nullable = true)
 |-- new_val: string (nullable = true)

